In [1]:
from os import listdir
from experiment_main import *

# ---------------------------------------------------------------------------------------
# Significance-test side channel (C7).
#
# deflated_sharpe needs two things that do not survive anywhere else: the SELECTED config's
# per-trade return series, and the sharpe of EVERY config that was searched (their spread
# sets the multiple-testing hurdle). naked_crossover computes the per-trade bets for all
# 475 configs on the validation window and get_crossover_results then discards them,
# keeping only the aggregates in dff_valid. This collects them instead.
#
# One record is appended per (crossover, system) run. Cleared by the caller before a run.
# ---------------------------------------------------------------------------------------
validation_bets = []


def get_crossover_results(
    df_test,
    df_valid,
    name_mode,
    target_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
):
    dff_test, current_bet_list_optim = naked_crossover(
        df_test,
        target_list,
        signal_line_list,
        trade_cost,
        balance,
        risk_per_trade=risk_per_trade,
        signal_line_if=signal_line_if,
        signal_line_if_target=signal_line_if_target,
        last_close=last_close,
    )

    check_index_first = find_best_index(
        dff_test, 1, "net", name_mode, alpha_range, gamma_range
    )
    check_index_second = find_best_index(
        dff_test, 2, "net", name_mode, alpha_range, gamma_range
    )
    check_index_third, best_list = find_best_index(
        dff_test, 3, "net", name_mode, alpha_range, gamma_range
    )
    check_index_fourth = find_best_index(
        dff_test, 4, "net", name_mode, alpha_range, gamma_range
    )  # Genetic algorithm
    # The long/short rule above picks a config by "net". A long-only trader would not pick
    # that config -- they would pick whichever config is best on LONG TRADES ALONE. So
    # select again on "long_only_net". This is what the main experiment already does
    # (experiment_main.py around line 3051) and what check_list_maker splices in as the
    # second long block; the case-study notebooks never did it, which is why their
    # "Long 2" columns were a duplicate of "Long 1" rather than a separate arm.
    check_index_first_long = find_best_index(
        dff_test, 1, "long_only_net", name_mode, alpha_range, gamma_range
    )
    check_index_second_long = find_best_index(
        dff_test, 2, "long_only_net", name_mode, alpha_range, gamma_range
    )
    check_index_third_long, best_list_long = find_best_index(
        dff_test, 3, "long_only_net", name_mode, alpha_range, gamma_range
    )
    check_index_fourth_long = find_best_index(
        dff_test, 4, "long_only_net", name_mode, alpha_range, gamma_range
    )  # Genetic algorithm

    base_histogram = dff_test[dff_test["his"] == f"{name_mode} a: 0"].index[0]

    # the base MACD row is the same config under either rule, so it is its own long-only arm
    long_indexes = [
        check_index_first_long,
        check_index_second_long,
        check_index_third_long,
        check_index_fourth_long,
        base_histogram,
    ]

    # print(check_index_first, check_index_second, check_index_third, check_index_fourth, base_histogram)

    dff_valid, current_bet_list_optim = naked_crossover(
        df_valid,
        target_list,
        signal_line_list,
        trade_cost,
        balance,
        risk_per_trade=risk_per_trade,
        signal_line_if=signal_line_if,
        signal_line_if_target=signal_line_if_target,
        last_close=last_close,
    )

    # stash every config's per-trade bets on the validation window, plus which configs
    # the four rules and the base picked, so the significance tests can be run afterwards
    # without re-running the backtest
    validation_bets.append(
        {
            "name_mode": name_mode,
            "signal_line_if": signal_line_if,
            "signal_line_if_target": signal_line_if_target,
            "chosen": {
                "rule_1": dff_valid.iloc[check_index_first]["his"],
                "rule_2": dff_valid.iloc[check_index_second]["his"],
                "rule_3": dff_valid.iloc[check_index_third]["his"],
                "rule_4": dff_valid.iloc[check_index_fourth]["his"],
                "base": dff_valid.iloc[base_histogram]["his"],
            },
            "bets": current_bet_list_optim,
        }
    )

    return (
        dff_valid,
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
        long_indexes,
    )

def extend_with_long_only(output_list, value_df, long_indexes):
    """
    extend_with_long_only appends the genuine long-only arm to each row, taking the long-only figures from the config the LONG-ONLY rule selected rather than from the config the long/short rule selected

    This replaces `row.extend(row[-3:])`, which simply duplicated the three long-only
    columns already on the row. That made the reported "Long 2" identical to "Long 1" by
    construction, so there was no long-only arm at all -- only the long leg of the
    long/short strategy, relabelled.

    Arguments:
        output_list -- the five result rows being assembled (first/second/third/fourth/base)
        value_df -- the validation results frame, whose columns 11:14 are long_only_net, long_only_bets and long_only_wins
        long_indexes -- the five row indexes chosen on "long_only_net", from get_crossover_results

    Returns:
        None -- extends each row of output_list in place
    """
    for position, long_index in enumerate(long_indexes):
        output_list[position].extend(list(value_df.iloc[long_index])[11:14])


def add_df_to_list_detailed(
    output_list,
    value_df,
    index_first,
    index_second,
    index_third,
    index_fourth,
    base_his,
):
    length_of_list = len(value_df.iloc[index_first]) - 1
    index_first_list = list(value_df.iloc[index_first])
    index_second_list = list(value_df.iloc[index_second])
    index_third_list = list(value_df.iloc[index_third])
    index_fourth_list = list(value_df.iloc[index_fourth])
    index_base_list = list(value_df.iloc[base_his])
    for i in range(length_of_list):
        output_list[0][i] += index_first_list[i]
        output_list[1][i] += index_second_list[i]
        output_list[2][i] += index_third_list[i]
        output_list[3][i] += index_fourth_list[i]
        output_list[4][i] += index_base_list[i]


def add_df_to_list_overall(output_list, value_df, index_first, base_his):
    length_of_list = len(value_df.iloc[index_first]) - 1
    index_first_list = list(value_df.iloc[index_first])
    index_base_list = list(value_df.iloc[base_his])
    for i in range(length_of_list):
        output_list[0][i] += index_first_list[i]
        output_list[1][i] += index_base_list[i]


def get_data_rule(rule_data, len_return_list):
    ticker_length = 1
    return_list = [[] for i in range(len_return_list)]

    [
        "net",  # 0
        "netbet",  # 1
        "netpercent",  # 2
        "bets",  # 3
        "wins",  # 4
        "longs",  # 5
        "long_wins",  # 6
        "long_profit",  # 7
        "shorts",  # 8
        "short_wins",  # 9
        "short_profit",  # 10
        "long_only_net",  # 11
        "long_only_bets",  # 12
        "long_only_wins",  # 13
        "long_only_net_other",  # 14
        "long_only_bets_other",  # 15
        "long_only_wins_other",  # 16
        "his",  # 17
    ]
    # [0'net' , 1'netbet' , 2'netpercent' , 3'bets' , 4'wins' , 5'longs' , 6'long_wins' , 7'long_profit' , 8'shorts' , 9'short_wins' , 10'short_profit',
    # 11"long_only_net", 12"long_only_bets", 13"long_only_wins", 14"long_only_net_other", 15"long_only_bets_other", 16"long_only_wins_other", 17'his']

    # avg return for ticker , bets , win/loss , avg long return for ticker , longs , long win ratio , avg short return , short win ratio,
    # 0 / tickers , 3 , 4/3 * % , 7 / tickers , 5 , 6 / 5 * % , 10 / tickers , 9 / 8 * %

    # average long only return for ticker, long only bets, long only win ratio, average other long return for ticker, other long bets, other long win ratio
    # 11 / tickers, 12, 13 / 12 * %, 14 / ticker, 15, 16 / 15 * %
    # print(len(obs_mode_data))
    # print(len(rule_data))
    for i in range(len(rule_data)):
        return_list[i].extend(
            [
                (rule_data[i][0]),  # / ticker_length)
                rule_data[i][3],
                (
                    (rule_data[i][4] * 100 / rule_data[i][3])
                    if rule_data[i][3] != 0
                    else 0
                ),
                (rule_data[i][7]),  # / ticker_length)
                rule_data[i][5],
                (
                    (rule_data[i][6] * 100 / rule_data[i][5])
                    if rule_data[i][5] != 0
                    else 0
                ),
                (rule_data[i][10]),  # / ticker_length)
                (
                    (rule_data[i][9] * 100 / rule_data[i][8])
                    if rule_data[i][8] != 0
                    else 0
                ),
                # Long 1 = the long leg of the config the LONG/SHORT rule picked
                (rule_data[i][11]),  # / ticker_length)
                rule_data[i][12],
                (
                    (rule_data[i][13] * 100 / rule_data[i][12])
                    if rule_data[i][12] != 0
                    else 0
                ),
                # Long 2 = the genuine long-only arm: the config the LONG-ONLY rule picked
                (rule_data[i][14]),  # / ticker_length)
                rule_data[i][15],
                (
                    (rule_data[i][16] * 100 / rule_data[i][15])
                    if rule_data[i][15] != 0
                    else 0
                ),
            ]
        )

    return return_list


def get_best_results(sys_crossover_list):
    best_results_index = 0
    for i in range(1, len(sys_crossover_list) - 1):
        # print(sys_crossover_list[i][0], sys_crossover_list[i-1][0])
        if sys_crossover_list[i][0] > sys_crossover_list[best_results_index][0]:
            best_results_index = i
    return sys_crossover_list[best_results_index]


def table_data_generator(data, indexes, len_return_list):
    output_data = []
    for d in data:
        current_return_list = get_data_rule(d, len_return_list)
        output_data.extend(current_return_list)

    # naked_cols = ['net', 'netbet', 'netpercent', 'bets', 'wins', 'longs', 'long_wins', 'long_profit',
    # 'shorts', 'short_wins', 'short_profit', "long_only_net", "long_only_bets", "long_only_wins",
    # "long_only_net_other", "long_only_bets_other", "long_only_wins_other", 'his']

    tempdf = pd.DataFrame(
        output_data,
        columns=[
            "Return",
            "Transactions",
            "Win Ratio",
            "Long Return",
            "Long Transactions",
            "Long Win Ratio",
            "Short Return",
            "Short Win Ratio",
            "Long 1 Return",
            "Long 1 Transactions",
            "Long 1 Win Ratio",
            "Long 2 Return",
            "Long 2 Transactions",
            "Long 2 Win Ratio",
        ],
        index=indexes,
    )

    return tempdf


alpha_range = [
    -0.9,
    -0.8,
    -0.7,
    -0.6,
    -0.5,
    -0.4,
    -0.3,
    -0.2,
    -0.1,
    0,
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
    0.9,
]
gamma_range = [
    1,
    2,
    4,
    10,
    20,
    30,
    40,
    50,
    60,
    70,
    80,
    90,
    100,
    200,
    300,
    400,
    500,
    600,
    700,
    800,
    900,
    1000,
    10000,
    100000,
]
histogram_list, macd_list, signal_line_list = return_macd_lists(
    alpha_range, gamma_range
)

In [ ]:
prefix_tex_tables = "/media/calavera/1TB HDD/Projects/MACD Project/Paper/paper_ready/table_image/"

market = "nasdaq"
prefix_text = f"./results/{market}"
file_name = f"insert_text_{market}.txt"
time_format = "1_day"

macd_params = (12, 26, 9)
# macd_params = (7, 10, 15)

# Aguirre validates on TWO separate windows (his Tables 2 and 3) and we only ever tested the
# first. Run this notebook once per window, the same way it is run once per macd_params.
validation_window = "initial"
# validation_window = "final"

mode = 'a'

# the window goes into params_str so every downstream results path and tex filename, which
# are all built from it, separates the two runs instead of overwriting one with the other
params_str = "_".join(str(val) for val in macd_params) + f"_{validation_window}"
short_period, long_period, signal_period = macd_params

# every (parameters, window) combination writes into its own results folder, and the
# 11-config sweep is about to add eleven more per market, so create the folder instead of
# requiring it to already exist. `from os import listdir` in cell 0 does not bind `os`.
import os

os.makedirs(f"./results/{market}/{params_str}", exist_ok=True)

trade_cost = False
balance = 5036790
risk_per_trade = 100 / 100
last_close = True

prefix = f"./data/{market}/"

ndq_read= pd.read_csv(
    prefix + "ndq.csv",
    index_col="Date",
    parse_dates=True,
    sep=",",
    usecols=["Date", "Open", "High", "Low", "Close"],
)

ndq_read.dropna(inplace=True)
ndq_read.columns = list(map(lambda x: x.lower(), list(ndq_read.columns)))
ndq_read.sort_index(inplace=True)
ndq_read.index.names = ["date"]

ndq_data = calculate_indicators(
    ndq_read,
    col="close",
    ema_mode="normal",
    short_period=short_period,
    long_period=long_period,
    signal_period=signal_period,
    alpha_range=alpha_range,
    gamma_range=gamma_range,
)

# the engine's strategies execute at a 'fill' column; without it macd_crossover raises KeyError.
# on daily bars every gap exceeds the session threshold, so fill == close, i.e. same-bar
# execution -- the same convention these published results were produced under.
add_fill_column(ndq_data)

ndq_data

,open,high,low,close,MACD,SignalLine,Histogram,MACD a: -0.9,MACD Signal a: -0.9,Histogram a: -0.9,...,Histogram a: 0.9 g: 1000,MACD a: 0.9 g: 1000 times,MACD a: 0.9 g: 10000,MACD Signal a: 0.9 g: 10000,Histogram a: 0.9 g: 10000,MACD a: 0.9 g: 10000 times,MACD a: 0.9 g: 100000,MACD Signal a: 0.9 g: 100000,Histogram a: 0.9 g: 100000,MACD a: 0.9 g: 100000 times
2009-07-01,1846.12,1861.62,1843.79,1845.72,18.917615,20.614165,-1.696551,21.791840,20.738860,1.052980,...,-4.362060,14306,26.490877,30.852937,-4.362060,17809,26.490877,30.852937,-4.362060,20703
2009-07-02,1823.69,1823.91,1795.95,1796.52,15.375644,19.566461,-4.190817,9.601068,18.511302,-8.910234,...,-4.095509,14306,25.733551,29.829060,-4.095509,17809,25.733551,29.829060,-4.095509,20703
2009-07-06,1783.50,1793.42,1770.11,1787.40,11.697854,17.992739,-6.294886,13.584961,17.526033,-3.941072,...,-4.105040,14306,24.697760,28.802800,-4.105040,17809,24.697760,28.802800,-4.105040,20703
2009-07-07,1787.44,1788.97,1745.38,1746.17,5.394082,15.473008,-10.078926,-1.977709,13.625285,-15.602994,...,-4.324024,14306,23.397770,27.721794,-4.324024,17809,23.397770,27.721794,-4.324024,20703
2009-07-08,1756.00,1757.67,1727.05,1747.17,0.473531,12.473113,-11.999582,2.679647,11.436157,-8.756511,...,-4.899514,14306,21.597401,26.496915,-4.899514,17809,21.597401,26.496915,-4.899514,20703
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-03-17,17722.55,17925.94,17645.88,17808.66,-522.928566,-448.025779,-74.902787,-530.004926,-456.705924,-73.299002,...,-112.052989,14306,-289.233846,-177.180858,-112.052989,17809,-289.233846,-177.180858,-112.052989,20703
2025-03-18,17662.57,17682.92,17431.67,17504.12,-524.731893,-463.367002,-61.364891,-519.986164,-469.361972,-50.624192,...,-108.337969,14306,-312.603318,-204.265350,-108.337969,17809,-312.603318,-204.265350,-108.337969,20703
2025-03-19,17590.12,17917.54,17533.94,17750.79,-500.487527,-470.791107,-29.696420,-482.938754,-472.077329,-10.861426,...,-103.640661,14306,-333.816176,-230.175515,-103.640661,17809,-333.816176,-230.175515,-103.640661,20703
2025-03-20,17586.06,17903.20,17576.82,17691.63,-480.508408,-472.734567,-7.773841,-478.321096,-473.326082,-4.995014,...,-96.246237,14306,-350.483311,-254.237074,-96.246237,17809,-350.483311,-254.237074,-96.246237,20703


In [5]:
time_jump = 24

# "initial" brackets Aguirre's initial validation, which runs from his first buy signal on
# 2015-08-21 to his last sell on 2017-10-05. "final" brackets his final validation,
# 2018-02-14 to 2019-02-25, which was never tested here before. We bracket his windows on
# clean quarter boundaries rather than copying his exact dates, because his endpoints are
# wherever his GA happened to signal, not a window definition.
if validation_window == "initial":
    end_optimize = pd.Timestamp("2015-06-30")
    start_test = pd.Timestamp("2015-07-01")
    end_test = pd.Timestamp("2017-12-31")
else:
    end_optimize = pd.Timestamp("2017-12-31")
    start_test = pd.Timestamp("2018-01-01")
    end_test = pd.Timestamp("2019-03-31")

start_optimize = make_new_time(end_optimize, -time_jump)

optimize_data = ndq_data[start_optimize:end_optimize]
test_data = ndq_data[start_test:end_test]

# Aguirre acquires 1,000 units at the start and reinvests the whole accumulated value at
# every subsequent purchase, so the starting balance is 1000x the index on the day HIS
# investment begins -- which is not the same as the day our test window opens.
#
# The 5,036,790 that was hardcoded here is not a magic number: it is exactly 1000x the
# close on 2015-08-11, the day the 2021 study's investment starts, and it is the initial
# investment his Table 3 reports. An earlier edit replaced it with 1000x the close at the
# test-window start (2015-07-01 -> 5,013,120), which quietly moved the anchor off his
# figure. Anchoring to each paper's own stated initial investment instead.
if validation_window == "initial":
    # 2021 JEFAS Table 3: investment begins 2015-08-11 at USD 5,036,790
    balance = 5036790
else:
    # 2020 IMFI Table 2: the final validation buys on 2018-02-14, 1,000 units at 7,083.65
    balance = 7083645

# optimize_data
test_data

,open,high,low,close,MACD,SignalLine,Histogram,MACD a: -0.9,MACD Signal a: -0.9,Histogram a: -0.9,...,Histogram a: 0.9 g: 1000,MACD a: 0.9 g: 1000 times,MACD a: 0.9 g: 10000,MACD Signal a: 0.9 g: 10000,Histogram a: 0.9 g: 10000,MACD a: 0.9 g: 10000 times,MACD a: 0.9 g: 100000,MACD Signal a: 0.9 g: 100000,Histogram a: 0.9 g: 100000,MACD a: 0.9 g: 100000 times
2015-07-01,5029.05,5038.40,4994.58,5013.12,-4.445838,10.107655,-14.553493,-10.568103,7.918457,-18.486561,...,-1.533026,14306,15.662311,17.195336,-1.533026,17809,15.662311,17.195336,-1.533026,20703
2015-07-02,5024.30,5027.18,4990.77,5009.21,-7.903079,6.505509,-14.408587,-5.504556,5.233855,-10.738411,...,-2.835072,14306,13.651496,16.486568,-2.835072,17809,13.651496,16.486568,-2.835072,20703
2015-07-06,4963.79,5020.61,4961.07,4991.94,-11.899343,2.824538,-14.723881,-17.654651,0.656154,-18.310805,...,-3.992424,14306,11.496038,15.488462,-3.992424,17809,11.496038,15.488462,-3.992424,20703
2015-07-07,4993.76,5001.95,4902.42,4997.46,-14.454373,-0.631244,-13.823129,-11.574122,-1.789902,-9.784220,...,-5.065570,14306,9.156500,14.222070,-5.065570,17809,9.156500,14.222070,-5.065570,20703
2015-07-08,4953.98,4965.33,4901.71,4909.76,-23.287466,-5.162488,-18.124977,-33.829475,-8.197816,-25.631659,...,-5.941326,14306,6.795413,12.736738,-5.941326,17809,6.795413,12.736738,-5.941326,20703
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2017-12-22,6958.02,6962.27,6944.45,6959.96,54.461027,48.559129,5.901898,53.746516,49.108382,4.638134,...,-0.172791,14306,47.299643,47.472434,-0.172791,17809,47.299643,47.472434,-0.172791,20703
2017-12-26,6928.93,6942.14,6915.56,6936.25,51.847653,49.216834,2.630819,50.138676,49.314441,0.824235,...,0.434678,14306,48.015782,47.581104,0.434678,17809,48.015782,47.581104,0.434678,20703
2017-12-27,6941.45,6955.38,6931.34,6939.34,49.455776,49.264622,0.191154,48.841167,49.219786,-0.378619,...,0.654292,14306,48.398969,47.744677,0.654292,17809,48.398969,47.744677,0.654292,20703
2017-12-28,6953.38,6954.81,6936.75,6950.16,47.881334,48.987965,-1.106630,47.017485,48.779326,-1.761841,...,0.607978,14306,48.504650,47.896671,0.607978,17809,48.504650,47.896671,0.607978,20703


In [9]:
# eyeball the bar on which each paper's investment actually begins. This was hardcoded to
# 2015-08-11, which only exists in the "initial" window and raised a KeyError on "final".
investment_start = (
    pd.Timestamp("2015-08-11")  # 2021 JEFAS: USD 5,036,790 goes in here
    if validation_window == "initial"
    else pd.Timestamp("2018-02-14")  # 2020 IMFI: the final validation's first buy
)
test_data.loc[investment_start]

open                             5069.380000
high                             5088.170000
low                              5013.570000
close                            5036.790000
MACD                               -1.602020
                                    ...     
MACD a: 0.9 g: 10000 times      17809.000000
MACD a: 0.9 g: 100000              12.206388
MACD Signal a: 0.9 g: 100000       13.080293
Histogram a: 0.9 g: 100000         -0.873904
MACD a: 0.9 g: 100000 times     20703.000000
Name: 2015-08-11 00:00:00, Length: 1888, dtype: float64

SYSTEM 1 - NO EXTRA CONDITION

In [4]:
signal_line_if = False
signal_line_if_target = False

Signalline crossover

In [5]:
sys1_major_macd_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "MACDCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    long_indexes,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    histogram_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys1_major_macd_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
# the long-only arm comes from the config the long-only rule chose, not from a
# copy of this row's own long columns
extend_with_long_only(sys1_major_macd_crossover, dff, long_indexes)
for i in range(len(sys1_major_macd_crossover)):
    print(sys1_major_macd_crossover[i], len(sys1_major_macd_crossover[i]))

best_sys1_major_macd_crossover = get_best_results(sys1_major_macd_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(-149366.14000000712), np.float64(-2297.940615384725), np.float64(-2.9655026316365607), np.int64(65), np.int64(27), np.int64(33), np.int64(21), np.float64(715063.6499999978), np.int64(32), np.int64(6), np.float64(-864429.7900000019), np.float64(813548.7299999995), np.int64(33), np.int64(21), np.float64(813548.7299999995), np.int64(33), np.int64(21)] 17
[np.float64(-511060.2700000042), np.float64(-10020.78960784322), np.float64(-10.146547106391257), np.int64(51), np.int64(20), np.int64(26), np.int64(15), np.float64(552620.2199999987), np.int64(25), np.int64(5), np.float64(-1063680.4900000019), np.float64(594226.1600000011), np.int64(26), np.int64(15), np.float64(594226.1600000011), np.int64(26), np.int64(15)] 17
[np.float64(-511060.2700000042), np.float64(-10020.78960784322), np.float64(-10.146547106391257), np.int64(51), np.int64(20), np.int64(26), np.int64(15), np.float64(552620.2199999987), np.int64(25), np.int64(5), np.float64(-1063680.4900000019), np.float64(594226.16000

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
8,-149366.14,-2297.940615,-2.965503,65,27,33,21,715063.65,32,6,-864429.79,813548.73,33,21,Histogram a: -0.1
16,-511060.27,-10020.789608,-10.146547,51,20,26,15,552620.22,25,5,-1063680.49,594226.16,26,15,Histogram a: 0.7
16,-511060.27,-10020.789608,-10.146547,51,20,26,15,552620.22,25,5,-1063680.49,594226.16,26,15,Histogram a: 0.7
368,28139.30,530.930189,0.558675,53,22,27,17,910005.96,26,5,-881866.66,908852.88,27,17,Histogram a: 0.5 g: 200
9,-219821.91,-3489.236667,-4.364325,63,26,32,20,674707.32,31,6,-894529.23,771587.78,32,20,Histogram a: 0


Zeroline crossover

In [6]:
sys1_major_zero_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "ZEROCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    long_indexes,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    macd_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys1_major_zero_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
# the long-only arm comes from the config the long-only rule chose, not from a
# copy of this row's own long columns
extend_with_long_only(sys1_major_zero_crossover, dff, long_indexes)
for i in range(len(sys1_major_zero_crossover)):
    print(sys1_major_zero_crossover[i], len(sys1_major_zero_crossover[i]))

best_sys1_major_zero_crossover = get_best_results(sys1_major_zero_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(252258.46000000183), np.float64(6306.461500000046), np.float64(5.008317996184114), np.int64(40), np.int64(11), np.int64(20), np.int64(9), np.float64(923209.3900000002), np.int64(20), np.int64(2), np.float64(-670950.9299999992), np.float64(941320.0500000017), np.int64(20), np.int64(9), np.float64(941320.0500000017), np.int64(20), np.int64(9)] 17
[np.float64(395765.4099999964), np.float64(26384.360666666427), np.float64(7.857492768211428), np.int64(15), np.int64(6), np.int64(7), np.int64(4), np.float64(909550.5499999989), np.int64(8), np.int64(2), np.float64(-513785.1400000009), np.float64(925097.1199999973), np.int64(7), np.int64(4), np.float64(925097.1199999973), np.int64(7), np.int64(4)] 17
[np.float64(-480497.77999999654), np.float64(-20891.207826086807), np.float64(-9.539762030976009), np.int64(23), np.int64(8), np.int64(11), np.int64(6), np.float64(359364.9100000006), np.int64(12), np.int64(2), np.float64(-839862.689999999), np.float64(414649.3200000003), np.int64(11), 

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
308,252258.46,6306.461500,5.008318,40,11,20,9,923209.39,20,2,-670950.93,941320.05,20,9,MACD a: 0.3 g: 2
18,395765.41,26384.360667,7.857493,15,6,7,4,909550.55,8,2,-513785.14,925097.12,7,4,MACD a: 0.9
427,-480497.78,-20891.207826,-9.539762,23,8,11,6,359364.91,12,2,-839862.69,414649.32,11,6,MACD a: 0.8 g: 1
433,-480497.78,-20891.207826,-9.539762,23,8,11,6,359364.91,12,2,-839862.69,414649.32,11,6,MACD a: 0.8 g: 40
9,90430.09,2055.229318,1.795391,44,14,22,10,931174.03,22,4,-840743.94,904471.78,22,10,MACD a: 0


SYSTEM 2 - SIGNAL EXTRA CONDITION

In [7]:
signal_line_if = True
signal_line_if_target = False

Signalline crossover

In [8]:
sys2_major_macd_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "MACDCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    long_indexes,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    histogram_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys2_major_macd_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
# the long-only arm comes from the config the long-only rule chose, not from a
# copy of this row's own long columns
extend_with_long_only(sys2_major_macd_crossover, dff, long_indexes)
for i in range(len(sys2_major_macd_crossover)):
    print(sys2_major_macd_crossover[i], len(sys2_major_macd_crossover[i]))

best_sys2_major_macd_crossover = get_best_results(sys2_major_macd_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(331262.6599999955), np.float64(19486.038823529147), np.float64(6.576860659269008), np.int64(17), np.int64(12), np.int64(9), np.int64(8), np.float64(837892.0399999989), np.int64(8), np.int64(4), np.float64(-506629.3800000005), np.float64(869070.7399999993), np.int64(9), np.int64(8), np.float64(869070.7399999993), np.int64(9), np.int64(8)] 17
[np.float64(-131682.6800000025), np.float64(-8778.8453333335), np.float64(-2.61441672176133), np.int64(15), np.int64(9), np.int64(8), np.int64(6), np.float64(699849.9199999989), np.int64(7), np.int64(3), np.float64(-831532.6000000015), np.float64(672543.6999999993), np.int64(8), np.int64(6), np.float64(672543.6999999993), np.int64(8), np.int64(6)] 17
[np.float64(-50099.210000001825), np.float64(-3339.947333333455), np.float64(-0.9946654516071114), np.int64(15), np.int64(10), np.int64(8), np.int64(7), np.float64(740056.3600000002), np.int64(7), np.int64(3), np.float64(-790155.5700000002), np.float64(694809.3500000006), np.int64(8), np.int

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
8,331262.66,19486.038824,6.576861,17,12,9,8,837892.04,8,4,-506629.38,869070.74,9,8,Histogram a: -0.1
16,-131682.68,-8778.845333,-2.614417,15,9,8,6,699849.92,7,3,-831532.60,672543.70,8,6,Histogram a: 0.7
379,-50099.21,-3339.947333,-0.994665,15,10,8,7,740056.36,7,3,-790155.57,694809.35,8,7,Histogram a: 0.6 g: 1
391,-167402.81,-11160.187333,-3.323601,15,9,8,7,658609.25,7,2,-826012.06,626285.10,8,7,Histogram a: 0.6 g: 100
9,287649.21,16920.541765,5.710963,17,12,9,8,809853.41,8,4,-522204.20,844769.39,9,8,Histogram a: 0


Zeroline crossover

In [9]:
sys2_major_zero_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "ZEROCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    long_indexes,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    macd_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys2_major_zero_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
# the long-only arm comes from the config the long-only rule chose, not from a
# copy of this row's own long columns
extend_with_long_only(sys2_major_zero_crossover, dff, long_indexes)
for i in range(len(sys2_major_zero_crossover)):
    print(sys2_major_zero_crossover[i], len(sys2_major_zero_crossover[i]))

best_sys2_major_zero_crossover = get_best_results(sys2_major_zero_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(107097.91000000015), np.float64(11899.767777777794), np.float64(2.1263127904875954), np.int64(9), np.int64(3), np.int64(5), np.int64(3), np.float64(475984.379999999), np.int64(4), np.int64(0), np.float64(-368886.470000001), np.float64(525431.5099999988), np.int64(5), np.int64(3), np.float64(525431.5099999988), np.int64(5), np.int64(3)] 17
[np.float64(-417912.90000000317), np.float64(-29850.921428571655), np.float64(-8.297207149791895), np.int64(14), np.int64(8), np.int64(7), np.int64(5), np.float64(253025.1300000007), np.int64(7), np.int64(3), np.float64(-670938.029999999), np.float64(501.7000000000007), np.int64(7), np.int64(5), np.float64(501.7000000000007), np.int64(7), np.int64(5)] 17
[np.float64(-417912.90000000317), np.float64(-29850.921428571655), np.float64(-8.297207149791895), np.int64(14), np.int64(8), np.int64(7), np.int64(5), np.float64(253025.1300000007), np.int64(7), np.int64(3), np.float64(-670938.029999999), np.float64(501.7000000000007), np.int64(7), np.int

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
379,107097.91,11899.767778,2.126313,9,3,5,3,475984.38,4,0,-368886.47,525431.51,5,3,MACD a: 0.6 g: 1
451,-417912.90,-29850.921429,-8.297207,14,8,7,5,253025.13,7,3,-670938.03,501.70,7,5,MACD a: 0.9 g: 1
451,-417912.90,-29850.921429,-8.297207,14,8,7,5,253025.13,7,3,-670938.03,501.70,7,5,MACD a: 0.9 g: 1
473,-417912.90,-29850.921429,-8.297207,14,8,7,5,253025.13,7,3,-670938.03,501.70,7,5,MACD a: 0.9 g: 10000
9,-410835.35,-24166.785294,-8.156690,17,6,9,4,393944.05,8,2,-804779.40,410429.90,9,4,MACD a: 0


SYSTEM 3 - Target SIGNAL EXTRA CONDITION

In [10]:
signal_line_if = False
signal_line_if_target = True

Signalline crossover

In [11]:
sys3_major_macd_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "MACDCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    long_indexes,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    histogram_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys3_major_macd_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
# the long-only arm comes from the config the long-only rule chose, not from a
# copy of this row's own long columns
extend_with_long_only(sys3_major_macd_crossover, dff, long_indexes)
for i in range(len(sys3_major_macd_crossover)):
    print(sys3_major_macd_crossover[i], len(sys3_major_macd_crossover[i]))

best_sys3_major_macd_crossover = get_best_results(sys3_major_macd_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(331262.6599999955), np.float64(19486.038823529147), np.float64(6.576860659269008), np.int64(17), np.int64(12), np.int64(9), np.int64(8), np.float64(837892.0399999989), np.int64(8), np.int64(4), np.float64(-506629.3800000005), np.float64(869070.7399999993), np.int64(9), np.int64(8), np.float64(869070.7399999993), np.int64(9), np.int64(8)] 17
[np.float64(-131682.6800000025), np.float64(-8778.8453333335), np.float64(-2.61441672176133), np.int64(15), np.int64(9), np.int64(8), np.int64(6), np.float64(699849.9199999989), np.int64(7), np.int64(3), np.float64(-831532.6000000015), np.float64(672543.6999999993), np.int64(8), np.int64(6), np.float64(672543.6999999993), np.int64(8), np.int64(6)] 17
[np.float64(-50099.210000001825), np.float64(-3339.947333333455), np.float64(-0.9946654516071114), np.int64(15), np.int64(10), np.int64(8), np.int64(7), np.float64(740056.3600000002), np.int64(7), np.int64(3), np.float64(-790155.5700000002), np.float64(694809.3500000006), np.int64(8), np.int

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
8,331262.66,19486.038824,6.576861,17,12,9,8,837892.04,8,4,-506629.38,869070.74,9,8,Histogram a: -0.1
16,-131682.68,-8778.845333,-2.614417,15,9,8,6,699849.92,7,3,-831532.60,672543.70,8,6,Histogram a: 0.7
379,-50099.21,-3339.947333,-0.994665,15,10,8,7,740056.36,7,3,-790155.57,694809.35,8,7,Histogram a: 0.6 g: 1
380,-50099.21,-3339.947333,-0.994665,15,10,8,7,740056.36,7,3,-790155.57,694809.35,8,7,Histogram a: 0.6 g: 2
9,287649.21,16920.541765,5.710963,17,12,9,8,809853.41,8,4,-522204.20,844769.39,9,8,Histogram a: 0


Zerocross crossover

In [12]:
sys3_major_zero_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "ZEROCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    long_indexes,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    macd_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys3_major_zero_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
# the long-only arm comes from the config the long-only rule chose, not from a
# copy of this row's own long columns
extend_with_long_only(sys3_major_zero_crossover, dff, long_indexes)
for i in range(len(sys3_major_zero_crossover)):
    print(sys3_major_zero_crossover[i], len(sys3_major_zero_crossover[i]))

best_sys3_major_zero_crossover = get_best_results(sys3_major_zero_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(-36041.870000001974), np.float64(-2120.110000000116), np.float64(-0.7155722196081626), np.int64(17), np.int64(8), np.int64(9), np.int64(6), np.float64(624885.8499999995), np.int64(8), np.int64(2), np.float64(-660927.72), np.float64(630180.8000000017), np.int64(9), np.int64(6), np.float64(630180.8000000017), np.int64(9), np.int64(6)] 17
[np.float64(-36041.870000001974), np.float64(-2120.110000000116), np.float64(-0.7155722196081626), np.int64(17), np.int64(8), np.int64(9), np.int64(6), np.float64(624885.8499999995), np.int64(8), np.int64(2), np.float64(-660927.72), np.float64(630180.8000000017), np.int64(9), np.int64(6), np.float64(630180.8000000017), np.int64(9), np.int64(6)] 17
[np.float64(-220739.64000000246), np.float64(-14715.976000000164), np.float64(-4.382546026338252), np.int64(15), np.int64(5), np.int64(8), np.int64(4), np.float64(557121.6499999989), np.int64(7), np.int64(1), np.float64(-777861.290000001), np.float64(635113.5699999984), np.int64(8), np.int64(4), np.

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
19,-36041.87,-2120.110000,-0.715572,17,8,9,6,624885.85,8,2,-660927.72,630180.80,9,6,MACD a: -0.9 g: 1
19,-36041.87,-2120.110000,-0.715572,17,8,9,6,624885.85,8,2,-660927.72,630180.80,9,6,MACD a: -0.9 g: 1
379,-220739.64,-14715.976000,-4.382546,15,5,8,4,557121.65,7,1,-777861.29,635113.57,8,4,MACD a: 0.6 g: 1
402,-220739.64,-14715.976000,-4.382546,15,5,8,4,557121.65,7,1,-777861.29,635113.57,8,4,MACD a: 0.6 g: 100000
9,-410835.35,-24166.785294,-8.156690,17,6,9,4,393944.05,8,2,-804779.40,410429.90,9,4,MACD a: 0


Getting Tables Detailed

Signalline crossover

In [13]:
# sys_modes = ["System 1", "System 2", "System 3"]
# # sys_modes = ["System 1", "System 2"]
# entries = ["Rule 1", "Rule 2", "Rule 3", "GA", "MACD"]
# tuples = []

# data = [sys1_major_macd_crossover, sys2_major_macd_crossover, sys3_major_macd_crossover]
# # data = [sys1_major_macd_crossover, sys2_major_macd_crossover]

# for sys_mode in sys_modes:
#     for entry in entries:
#         tuples.append((sys_mode, entry))
# table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

# table_data = table_data_generator(data, table_index, 5)

# columns = [
#     "Return",
#     "Win Ratio",
#     "Long Return",
#     "Long Win Ratio",
#     "Short Return",
#     "Short Win Ratio",
#     # "Long 1 Return",
#     # "Long 1 Win Ratio",
#     # "Long 2 Return",
#     # "Long 2 Win Ratio",
# ]
# for col in columns:
#     l = []
#     for x in table_data[col]:
#         l.append(round(x, 2))
#     table_data[col] = l

# table_data.to_csv(f"./results/nasdaq/{params_str}/nasdaq_macdcross.csv", index=True)

# table_data

In [14]:
# # sys_modes = ["System 1", "System 2", "System 3"]
# sys_modes = ["System 1", "System 2"]
# entries = ["Best", "MACD"]
# tuples = []

# # data = [[best_sys1_major_macd_crossover, sys1_major_macd_crossover[-1]], [best_sys2_major_macd_crossover, sys2_major_macd_crossover[-1]], [best_sys3_major_macd_crossover, sys3_major_macd_crossover[-1]]]
# data = [
#     [best_sys1_major_macd_crossover, sys1_major_macd_crossover[-1]],
#     [best_sys2_major_macd_crossover, sys2_major_macd_crossover[-1]],
# ]

# for sys_mode in sys_modes:
#     for entry in entries:
#         tuples.append((sys_mode, entry))
# table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

# table_data = table_data_generator(data, table_index, 2)

# columns = [
#     "Return",
#     "Win Ratio",
#     "Long Return",
#     "Long Win Ratio",
#     "Short Return",
#     "Short Win Ratio",
#     # "Long 1 Return",
#     # "Long 1 Win Ratio",
#     # "Long 2 Return",
#     # "Long 2 Win Ratio",
# ]
# for col in columns:
#     l = []
#     for x in table_data[col]:
#         l.append(round(x, 2))
#     table_data[col] = l

# table_data.to_csv(
#     f"./results/nasdaq/{params_str}/nasdaq_macdcross_overall.csv", index=True
# )

# table_data

Zeroline crossover

In [15]:
# sys_modes = ["System 1", "System 2", "System 3"]
# # sys_modes = ["System 1", "System 2"]
# entries = ["Rule 1", "Rule 2", "Rule 3", "GA", "MACD"]
# tuples = []

# data = [sys1_major_zero_crossover, sys2_major_zero_crossover, sys3_major_zero_crossover]
# # data = [sys1_major_zero_crossover, sys2_major_zero_crossover]

# for sys_mode in sys_modes:
#     for entry in entries:
#         tuples.append((sys_mode, entry))
# table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

# table_data = table_data_generator(data, table_index, 5)

# columns = [
#     "Return",
#     "Win Ratio",
#     "Long Return",
#     "Long Win Ratio",
#     "Short Return",
#     "Short Win Ratio",
#     # "Long 1 Return",
#     # "Long 1 Win Ratio",
#     # "Long 2 Return",
#     # "Long 2 Win Ratio",
# ]
# for col in columns:
#     l = []
#     for x in table_data[col]:
#         l.append(round(x, 2))
#     table_data[col] = l

# table_data.to_csv(f"./results/nasdaq/{params_str}/nasdaq_zerocross.csv", index=True)

# table_data

In [16]:
# # sys_modes = ["System 1", "System 2", "System 3"]
# sys_modes = ["System 1", "System 2"]
# entries = ["Best", "MACD"]
# tuples = []

# # data = [[best_sys1_major_zero_crossover, sys1_major_zero_crossover[-1]], [best_sys2_major_zero_crossover, sys2_major_zero_crossover[-1]], [best_sys3_major_zero_crossover, sys3_major_zero_crossover[-1]]]
# data = [
#     [best_sys1_major_zero_crossover, sys1_major_zero_crossover[-1]],
#     [best_sys2_major_zero_crossover, sys2_major_zero_crossover[-1]],
# ]

# for sys_mode in sys_modes:
#     for entry in entries:
#         tuples.append((sys_mode, entry))
# table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

# table_data = table_data_generator(data, table_index, 2)

# columns = [
#     "Return",
#     "Win Ratio",
#     "Long Return",
#     "Long Win Ratio",
#     "Short Return",
#     "Short Win Ratio",
#     # "Long 1 Return",
#     # "Long 1 Win Ratio",
#     # "Long 2 Return",
#     # "Long 2 Win Ratio",
# ]
# for col in columns:
#     l = []
#     for x in table_data[col]:
#         l.append(round(x, 2))
#     table_data[col] = l

# table_data.to_csv(
#     f"./results/nasdaq/{params_str}/nasdaq_zerocross_overall.csv", index=True
# )

# table_data

In [17]:
crossover_modes = ["MACD Crossover", "Zero Crossover"]
sys_modes = ["System 1", "System 2", "System 3"]
entries = ["Rule 1", "Rule 2", "Rule 3", "GA", "MACD"]
tuples = []

data = [sys1_major_macd_crossover, sys2_major_macd_crossover, sys3_major_macd_crossover,
        sys1_major_zero_crossover, sys2_major_zero_crossover, sys3_major_zero_crossover]
# data = [sys1_major_macd_crossover, sys2_major_macd_crossover]

for crossover_mode in crossover_modes:
    for sys_mode in sys_modes:
        for entry in entries:
            tuples.append((crossover_mode, sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second", "third"])

table_data = table_data_generator(data, table_index, 5)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(f"./results/nasdaq/{params_str}/nasdaq_all.csv", index=True)
caption = f"Results for {macd_params} paramters for MACD by applying the simulation scenario on \\citet{"{Aguirre2020}"}'s data"
write_table_tex(table_data, f"./results/nasdaq/{params_str}", "nasdaq_all", f"nasdaq_{params_str}_all", caption)
write_table_tex(table_data, prefix_tex_tables, f"nasdaq_all_{params_str}", f"nasdaq_{params_str}_all", caption)

text = f"\\input{"{"}nasdaq_all_{params_str}_tex{"}"}"
if macd_params == (12, 26, 9):
    write_to_text(prefix_text, file_name, text, mode='w')
else:
    write_to_text(prefix_text, file_name, text, mode='a')

table_data

Return  Transactions  Win Ratio  \
first          second   third                                        
MACD Crossover System 1 Rule 1 -149366.14            65      41.54   
                        Rule 2 -511060.27            51      39.22   
                        Rule 3 -511060.27            51      39.22   
                        GA       28139.30            53      41.51   
                        MACD   -219821.91            63      41.27   
               System 2 Rule 1  331262.66            17      70.59   
                        Rule 2 -131682.68            15      60.00   
                        Rule 3  -50099.21            15      66.67   
                        GA     -167402.81            15      60.00   
                        MACD    287649.21            17      70.59   
               System 3 Rule 1  331262.66            17      70.59   
                        Rule 2 -131682.68            15      60.00   
                        Rule 3  -50099.21            15      66.67   
                        GA      -50099.21            15      66.67   
                        MACD    287649.21            17      70.59   
Zero Crossover System 1 Rule 1  252258.46            40      27.50   
                        Rule 2  395765.41            15      40.00   
                        Rule 3 -480497.78            23      34.78   
                        GA     -480497.78            23      34.78   
                        MACD     90430.09            44      31.82   
               System 2 Rule 1  107097.91             9      33.33   
                        Rule 2 -417912.90            14      57.14   
                        Rule 3 -417912.90            14      57.14   
                        GA     -417912.90            14      57.14   
                        MACD   -410835.35            17      35.29   
               System 3 Rule 1  -36041.87            17      47.06   
                        Rule 2  -36041.87            17      47.06   
                        Rule 3 -220739.64            15      33.33   
                        GA     -220739.64            15      33.33   
                        MACD   -410835.35            17      35.29   

                                Long Return  Long Transactions  \
first          second   third                                    
MACD Crossover System 1 Rule 1    715063.65                 33   
                        Rule 2    552620.22                 26   
                        Rule 3    552620.22                 26   
                        GA        910005.96                 27   
                        MACD      674707.32                 32   
               System 2 Rule 1    837892.04                  9   
                        Rule 2    699849.92                  8   
                        Rule 3    740056.36                  8   
                        GA        658609.25                  8   
                        MACD      809853.41                  9   
               System 3 Rule 1    837892.04                  9   
                        Rule 2    699849.92                  8   
                        Rule 3    740056.36                  8   
                        GA        740056.36                  8   
                        MACD      809853.41                  9   
Zero Crossover System 1 Rule 1    923209.39                 20   
                        Rule 2    909550.55                  7   
                        Rule 3    359364.91                 11   
                        GA        359364.91                 11   
                        MACD      931174.03                 22   
               System 2 Rule 1    475984.38                  5   
                        Rule 2    253025.13                  7   
                        Rule 3    253025.13                  7   
                        GA        253025.13                  7   
                        MACD      393944.05                  9   
            

In [ ]:
crossover_modes = ["MACD Crossover", "Zero Crossover"]
sys_modes = ["System 1", "System 2"]
entries = ["Best", "MACD"]
tuples = []

# data = [[best_sys1_major_macd_crossover, sys1_major_macd_crossover[-1]], [best_sys2_major_macd_crossover, sys2_major_macd_crossover[-1]], [best_sys3_major_macd_crossover, sys3_major_macd_crossover[-1]]]
data = [
    [best_sys1_major_macd_crossover, sys1_major_macd_crossover[-1]],
    [best_sys2_major_macd_crossover, sys2_major_macd_crossover[-1]],
    [best_sys1_major_zero_crossover, sys1_major_zero_crossover[-1]],
    [best_sys2_major_zero_crossover, sys2_major_zero_crossover[-1]],
]

for crossover_mode in crossover_modes:
    for sys_mode in sys_modes:
        for entry in entries:
            tuples.append((crossover_mode, sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second", "third"])

table_data = table_data_generator(data, table_index, 2)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # "Long 1 Return",
    # "Long 1 Win Ratio",
    # "Long 2 Return",
    # "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(
    f"./results/nasdaq/{params_str}/nasdaq_overall.csv", index=True
)

# The csv keeps all fourteen columns. The tex table takes a chosen subset, because
# "Long 1" is the long LEG of the long/short strategy and is structurally the same as
# "Long Return" for an always-in-market crossover system, so printing both would just
# repeat a column. "Long 2" is the separate long-only strategy and is the one worth
# showing. The first eight must stay in this order: the published comparison rows line
# up against them.
tex_columns = [
    "Return",
    "Transactions",
    "Win Ratio",
    "Long Return",
    "Long Transactions",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    "Long 2 Return",
    "Long 2 Transactions",
    "Long 2 Win Ratio",
]
tex_data = table_data[tex_columns].copy()

# Aguirre reports an effective annual rate in percent, not an amount, so the shared
# "Return" header means something different in this table than in the other two. This adds
# his metric alongside ours, computed HIS way -- see aguirre_annual_rate for why we copy a
# convention we know understates both sides rather than using the correct one.
#
# DECISION C5: WHICH way is "his" depends on WHICH paper the window is cited against, and
# the two do not share a convention. The 2021 JEFAS paper annualises a log return as
# though it were simple; the 2020 IMFI paper uses an ordinary CAGR on a simple nominal.
# Computing both windows the 2021 way, as this line did, understates us by 2.61pp on his
# initial window and 1.72pp on his final one measured against the 2020 paper's own basis.
# That direction runs against our own result, but it is still a like-for-unlike comparison
# in the single column that claims to be like-for-like. The branch matches the
# aguirre_paper branch below: initial -> 2021 JEFAS, final -> 2020 IMFI.
annual_convention = "log_as_simple" if validation_window == "initial" else "cagr"
investment_days = (test_data.index.max() - test_data.index.min()).days
tex_data["Annual %"] = [
    aguirre_annual_rate(net, balance, investment_days, convention=annual_convention)
    for net in tex_data["Return"]
]

if macd_params == (12, 26, 9):
    # the two Aguirre papers are DIFFERENT studies, so each window cites the one that
    # actually covers it. The 2021 JEFAS study validates over 2015-07-01..2017-12-31, which
    # is this notebook's "initial" window and the study this notebook was built against
    # (its balance, window and 7_10_15 parameters are all his). The 2020 IMFI study's
    # second validation window is what our "final" window covers.
    aguirre_paper = "aguirre2021" if validation_window == "initial" else "aguirre2020"
    aguirre_cite = "Aguirre2021" if validation_window == "initial" else "Aguirre2020"
    caption = f"Comparison between the results published in \\citet{{{aguirre_cite}}} and the results obtained by applying the simulation scenario on their data"
    write_overall_table_tex(tex_data, f"./results/nasdaq/{params_str}", "nasdaq_overall", f"nasdaq_{params_str}_overall", caption, aguirre_paper)
    write_overall_table_tex(tex_data, prefix_tex_tables, f"nasdaq_overall_{params_str}", f"nasdaq_{params_str}_overall", caption, aguirre_paper)
else:
    write_overall_table_tex(tex_data, f"./results/nasdaq/{params_str}", "nasdaq_overall", f"nasdaq_{params_str}_overall", caption, None)
    write_overall_table_tex(tex_data, prefix_tex_tables, f"nasdaq_overall_{params_str}", f"nasdaq_{params_str}_overall", caption, None)
text = f"\\input{"{"}nasdaq_overall_{params_str}_tex{"}"}"
write_to_text(prefix_text, file_name, text, mode=mode)
table_data

Return  Transactions  Win Ratio  \
first          second   third                                       
MACD Crossover System 1 Best    28139.30            53      41.51   
                        MACD  -219821.91            63      41.27   
               System 2 Best   331262.66            17      70.59   
                        MACD   287649.21            17      70.59   
Zero Crossover System 1 Best   395765.41            15      40.00   
                        MACD    90430.09            44      31.82   
               System 2 Best   107097.91             9      33.33   
                        MACD  -410835.35            17      35.29   

                               Long Return  Long Transactions  Long Win Ratio  \
first          second   third                                                   
MACD Crossover System 1 Best     910005.96                 27           62.96   
                        MACD     674707.32                 32           62.50   
               System 2 Best     837892.04                  9           88.89   
                        MACD     809853.41                  9           88.89   
Zero Crossover System 1 Best     909550.55                  7           57.14   
                        MACD     931174.03                 22           45.45   
               System 2 Best     475984.38                  5           60.00   
                        MACD     393944.05                  9           44.44   

                               Short Return  Short Win Ratio  
first          second   third                                 
MACD Crossover System 1 Best     -881866.66            19.23  
                        MACD     -894529.23            19.35  
               System 2 Best     -506629.38            50.00  
                        MACD     -522204.20            50.00  
Zero Crossover System 1 Best     -513785.14            25.00  
                        MACD     -840743.94            18.18  
               System 2 Best     -368886.47             0.00  
                        MACD     -804779.40            25.00